# Lecture 9 Examples and Case — Automation and Agent Workflows

**Course:** AAU E26 — Introduction to Scripting, Data Mining and Machine Learning  
**Lecture:** Lecture 9  
**Goal:** Compare fixed automation with a bounded, auditable, human-controlled workflow.

[Open this notebook in Google Colab](https://colab.research.google.com/github/asmrabbi/E26_TAN7_Scripting_CPH/blob/main/notebooks/examples/L09_examples_automation_agent_workflows.ipynb) · [View the course repository](https://github.com/asmrabbi/E26_TAN7_Scripting_CPH)

Run the cells from top to bottom. Every executable line includes a short comment explaining what it does.


## Goal

Compare a deterministic automation with a bounded agent-like workflow that uses tools, logs actions, and stops for human approval.


## Setup

This notebook is a safe simulation. It does not send messages, upload files, or call external AI services.


## Steps

### 1. Define a conventional automation


In [1]:
def classify_report_status(missing_cells, rule_failures):  # Defines a deterministic function with two explicit inputs.
    """Return a status using fixed rules that do not learn or improvise."""  # Documents the function in ordinary language.
    if missing_cells > 0 or rule_failures > 0:  # Checks whether any stated data-quality problem exists.
        return "needs review"  # Returns the same status whenever the condition is true.
    return "ready"  # Returns the alternative status when no stated problem exists.
print(classify_report_status(0, 0))  # Tests the normal ready case.
print(classify_report_status(2, 0))  # Tests a case with missing information.


ready
needs review


### 2. Define small read-only tools


In [2]:
def count_missing_values(report):  # Defines a tool that counts fields whose value is None.
    """Count missing values in one dictionary-shaped report."""  # Documents the tool output.
    return sum(value is None for value in report.values())  # Returns the number of missing values without changing the report.
def calculate_resolution_rate(report):  # Defines a tool that calculates a rate from two report fields.
    """Return a resolution rate or None when calculation is unsafe."""  # Documents the result and its possible missing state.
    received = report.get("cases_received")  # Retrieves the denominator safely.
    resolved = report.get("cases_resolved")  # Retrieves the numerator safely.
    if received in (None, 0) or resolved is None:  # Checks for missing values or division by zero.
        return None  # Stops the calculation when the required inputs are unsafe.
    return resolved / received  # Returns the proportion for valid inputs.


### 3. Run a bounded agent-like workflow


In [3]:
def review_report(report, maximum_steps=3):  # Defines a workflow with a clear input and step limit.
    """Use read-only tools, log results, and request human review when needed."""  # Documents the workflow behaviour.
    action_log = []  # Creates an audit trail for tool use and decisions.
    missing_count = count_missing_values(report)  # Calls the first read-only tool.
    action_log.append(f"count_missing_values -> {missing_count}")  # Records the first tool result.
    resolution_rate = calculate_resolution_rate(report)  # Calls the second read-only tool.
    action_log.append(f"calculate_resolution_rate -> {resolution_rate}")  # Records the second tool result.
    if len(action_log) >= maximum_steps:  # Checks the explicit stopping condition before any extra work.
        return {"status": "stopped at step limit", "log": action_log}  # Stops and returns the visible audit trail.
    if missing_count > 0 or resolution_rate is None or resolution_rate > 1:  # Checks whether a person must inspect the input or result.
        return {"status": "human approval required", "log": action_log}  # Stops before any external action can occur.
    return {"status": "ready for human-approved next step", "log": action_log}  # Reports readiness without sending or publishing anything.


### 4. Test normal and unexpected inputs


In [4]:
normal_report = {"record_id": 1001, "cases_received": 120, "cases_resolved": 112}  # Stores a report with complete plausible counts.
missing_report = {"record_id": 1002, "cases_received": 85, "cases_resolved": None}  # Stores a report with a missing required value.
contradictory_report = {"record_id": 1003, "cases_received": 85, "cases_resolved": 90}  # Stores a report that violates the simplified count rule.
for test_report in [normal_report, missing_report, contradictory_report]:  # Runs the same bounded workflow on three different cases.
    review_result = review_report(test_report)  # Collects the workflow status and audit log.
    print(test_report["record_id"], review_result)  # Displays the input identifier beside the complete result.


1001 {'status': 'ready for human-approved next step', 'log': ['count_missing_values -> 0', 'calculate_resolution_rate -> 0.9333333333333333']}
1002 {'status': 'human approval required', 'log': ['count_missing_values -> 1', 'calculate_resolution_rate -> None']}
1003 {'status': 'human approval required', 'log': ['count_missing_values -> 0', 'calculate_resolution_rate -> 1.0588235294117647']}


### 5. Compare the two approaches


In [5]:
comparison = {"fixed automation": "applies predetermined rules and returns a status", "agent-like workflow": "selects from named tools, records actions, obeys a step limit, and stops for approval"}  # Summarises the operational difference shown in the notebook.
for approach_name, approach_description in comparison.items():  # Visits each comparison entry.
    print(f"{approach_name}: {approach_description}")  # Displays the comparison in plain language.


fixed automation: applies predetermined rules and returns a status
agent-like workflow: selects from named tools, records actions, obeys a step limit, and stops for approval


## Checks

No external side effect occurs. The workflow exposes its tool results, obeys a maximum-step condition, and sends ambiguous or contradictory data to a person.


## Next Steps

Use the exercise notebook to add one safe read-only tool, one unexpected test case, and one explicit approval point.
